In [2]:
!pip -q install datasets

In [3]:
import re
import numpy as np
import pandas as pd
from collections import Counter
from datasets import load_dataset

In [4]:
LANGUAGES = {
    "asm_Beng": "Assamese",
    "ben_Beng": "Bengali",
    "brx_Deva": "Bodo",
    "doi_Deva": "Dogri",
    "gom_Deva": "Konkani",
    "guj_Gujr": "Gujarati",
    "hin_Deva": "Hindi",
    "kan_Knda": "Kannada",
    "kas_Arab": "Kashmiri",
    "mai_Deva": "Maithili",
    "mal_Mlym": "Malayalam",
    "mar_Deva": "Marathi",
    "mni_Mtei": "Manipuri",
    "npi_Deva": "Nepali",
    "ory_Orya": "Odia",
    "pan_Guru": "Punjabi",
    "san_Deva": "Sanskrit",
    "snd_Deva": "Sindhi",
    "tam_Taml": "Tamil",
    "tel_Telu": "Telugu",
    "urd_Arab": "Urdu",
    "khasi": "Khasi",
    "santhali": "Santali"
}

print("Number of languages:", len(LANGUAGES))

Number of languages: 23


In [5]:
def sentence_tokenize(text):
    text = str(text).strip()

    # Indian + common sentence-ending punctuation
    sentences = re.split(r'(?<=[.!?।॥])\s+', text)

    return [
        s.strip()
        for s in sentences
        if len(s.strip()) >= 15
    ]

In [6]:
def collect_sentences(lang_code, n=1000):
    print(f"Loading {LANGUAGES[lang_code]}...")

    ds = load_dataset(
        "ai4bharat/IndicCorpV2",
        "indiccorp_v2",
        split=lang_code,
        streaming=True
    )

    sentences = []

    for row in ds:
        text = row["text"]

        for sent in sentence_tokenize(text):
            sentences.append(sent)

            if len(sentences) >= n:
                break

        if len(sentences) >= n:
            break

    print(f"Collected: {len(sentences)} sentences")
    return sentences

In [7]:
all_texts = []
all_labels = []

for code in LANGUAGES:
    sentences = collect_sentences(code, 1000)

    all_texts.extend(sentences)
    all_labels.extend([code] * len(sentences))

print("\nTotal sentences:", len(all_texts))
print("Total labels:", len(all_labels))

Loading Assamese...


README.md:   0%|          | 0.00/4.01k [00:00<?, ?B/s]

Collected: 1000 sentences
Loading Bengali...
Collected: 1000 sentences
Loading Bodo...
Collected: 1000 sentences
Loading Dogri...
Collected: 1000 sentences
Loading Konkani...
Collected: 1000 sentences
Loading Gujarati...
Collected: 1000 sentences
Loading Hindi...
Collected: 1000 sentences
Loading Kannada...
Collected: 1000 sentences
Loading Kashmiri...
Collected: 1000 sentences
Loading Maithili...
Collected: 1000 sentences
Loading Malayalam...
Collected: 1000 sentences
Loading Marathi...
Collected: 1000 sentences
Loading Manipuri...
Collected: 1000 sentences
Loading Nepali...
Collected: 1000 sentences
Loading Odia...
Collected: 1000 sentences
Loading Punjabi...
Collected: 1000 sentences
Loading Sanskrit...
Collected: 1000 sentences
Loading Sindhi...
Collected: 1000 sentences
Loading Tamil...
Collected: 1000 sentences
Loading Telugu...
Collected: 1000 sentences
Loading Urdu...
Collected: 1000 sentences
Loading Khasi...
Collected: 1000 sentences
Loading Santali...
Collected: 1000 sentenc

In [8]:
counts = Counter(all_labels)

for code, count in counts.items():
    print(f"{LANGUAGES[code]:12s} : {count}")

Assamese     : 1000
Bengali      : 1000
Bodo         : 1000
Dogri        : 1000
Konkani      : 1000
Gujarati     : 1000
Hindi        : 1000
Kannada      : 1000
Kashmiri     : 1000
Maithili     : 1000
Malayalam    : 1000
Marathi      : 1000
Manipuri     : 1000
Nepali       : 1000
Odia         : 1000
Punjabi      : 1000
Sanskrit     : 1000
Sindhi       : 1000
Tamil        : 1000
Telugu       : 1000
Urdu         : 1000
Khasi        : 1000
Santali      : 1000


In [9]:
np.random.seed(42)

train_texts, val_texts, test_texts = [], [], []
train_labels, val_labels, test_labels = [], [], []

for lang in LANGUAGES:
    idx = [i for i, x in enumerate(all_labels) if x == lang]
    np.random.shuffle(idx)

    train_idx = idx[:800]
    val_idx = idx[800:900]
    test_idx = idx[900:1000]

    for i in train_idx:
        train_texts.append(all_texts[i])
        train_labels.append(lang)

    for i in val_idx:
        val_texts.append(all_texts[i])
        val_labels.append(lang)

    for i in test_idx:
        test_texts.append(all_texts[i])
        test_labels.append(lang)

print("Train:", len(train_texts))
print("Validation:", len(val_texts))
print("Test:", len(test_texts))

Train: 18400
Validation: 2300
Test: 2300


In [10]:
def shuffle_data(texts, labels):
    idx = np.random.permutation(len(texts))
    return [texts[i] for i in idx], [labels[i] for i in idx]

train_texts, train_labels = shuffle_data(train_texts, train_labels)
val_texts, val_labels = shuffle_data(val_texts, val_labels)
test_texts, test_labels = shuffle_data(test_texts, test_labels)

In [11]:
class MyTfidf:

    def __init__(self, max_features=12000):
        self.max_features = max_features
        self.vocab = {}
        self.idf = None

    def features(self, text):
        text = text.lower()

        # Word tokens
        words = re.findall(r'\S+', text)

        feats = []

        # Word unigrams
        feats.extend(words)

        # Word bigrams
        feats.extend(
            words[i] + "_" + words[i+1]
            for i in range(len(words)-1)
        )

        # Character n-grams
        compact = re.sub(r'\s+', ' ', text)

        for n in [2, 3, 4]:
            feats.extend(
                compact[i:i+n]
                for i in range(len(compact)-n+1)
            )

        return feats

    def fit(self, texts):
        df = Counter()

        for text in texts:
            unique_features = set(self.features(text))
            df.update(unique_features)

        # Keep most frequent features
        common = df.most_common(self.max_features)

        self.vocab = {
            feature: i
            for i, (feature, _) in enumerate(common)
        }

        N = len(texts)

        self.idf = np.ones(len(self.vocab), dtype=np.float32)

        for feature, i in self.vocab.items():
            self.idf[i] = np.log(
                (N + 1) / (df[feature] + 1)
            ) + 1

        print("Vocabulary size:", len(self.vocab))

    def transform(self, texts):
        X = np.zeros(
            (len(texts), len(self.vocab)),
            dtype=np.float32
        )

        for row, text in enumerate(texts):
            counts = Counter(self.features(text))

            total = sum(
                count for feature, count in counts.items()
                if feature in self.vocab
            )

            if total == 0:
                continue

            for feature, count in counts.items():
                if feature in self.vocab:
                    j = self.vocab[feature]
                    tf = count / total
                    X[row, j] = tf * self.idf[j]

        # L2 normalization
        norms = np.sqrt((X * X).sum(axis=1, keepdims=True))
        X /= np.maximum(norms, 1e-8)

        return X

In [12]:
vectorizer = MyTfidf(max_features=12000)

vectorizer.fit(train_texts)

Vocabulary size: 12000


In [13]:
X_train = vectorizer.transform(train_texts)
X_val = vectorizer.transform(val_texts)
X_test = vectorizer.transform(test_texts)

print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)

Train shape: (18400, 12000)
Validation shape: (2300, 12000)
Test shape: (2300, 12000)


In [14]:
label_names = list(LANGUAGES.keys())

label_to_id = {
    label: i
    for i, label in enumerate(label_names)
}

id_to_label = {
    i: label
    for label, i in label_to_id.items()
}

y_train = np.array([label_to_id[x] for x in train_labels])
y_val = np.array([label_to_id[x] for x in val_labels])
y_test = np.array([label_to_id[x] for x in test_labels])

print(label_to_id)

{'asm_Beng': 0, 'ben_Beng': 1, 'brx_Deva': 2, 'doi_Deva': 3, 'gom_Deva': 4, 'guj_Gujr': 5, 'hin_Deva': 6, 'kan_Knda': 7, 'kas_Arab': 8, 'mai_Deva': 9, 'mal_Mlym': 10, 'mar_Deva': 11, 'mni_Mtei': 12, 'npi_Deva': 13, 'ory_Orya': 14, 'pan_Guru': 15, 'san_Deva': 16, 'snd_Deva': 17, 'tam_Taml': 18, 'tel_Telu': 19, 'urd_Arab': 20, 'khasi': 21, 'santhali': 22}


In [15]:
class MyLogisticRegression:

    def __init__(self, learning_rate=0.5, epochs=30):
        self.lr = learning_rate
        self.epochs = epochs
        self.W = None
        self.b = None

    def softmax(self, z):
        z = z - np.max(z, axis=1, keepdims=True)
        exp_z = np.exp(z)
        return exp_z / np.sum(exp_z, axis=1, keepdims=True)

    def fit(self, X, y, X_val=None, y_val=None):

        n_samples, n_features = X.shape
        n_classes = len(np.unique(y))

        self.W = np.zeros(
            (n_features, n_classes),
            dtype=np.float32
        )

        self.b = np.zeros(
            n_classes,
            dtype=np.float32
        )

        Y = np.zeros(
            (n_samples, n_classes),
            dtype=np.float32
        )
        Y[np.arange(n_samples), y] = 1

        for epoch in range(self.epochs):

            scores = X @ self.W + self.b
            probs = self.softmax(scores)

            error = probs - Y

            grad_W = (X.T @ error) / n_samples
            grad_b = np.mean(error, axis=0)

            self.W -= self.lr * grad_W
            self.b -= self.lr * grad_b

            if (epoch + 1) % 5 == 0:
                pred = np.argmax(probs, axis=1)
                acc = np.mean(pred == y)

                print(
                    f"Epoch {epoch+1}/{self.epochs} "
                    f"- Train Accuracy: {acc:.4f}"
                )

    def predict(self, X):
        scores = X @ self.W + self.b
        return np.argmax(scores, axis=1)

In [16]:
model = MyLogisticRegression(
    learning_rate=0.5,
    epochs=20
)

model.fit(X_train, y_train)

Epoch 5/20 - Train Accuracy: 0.9043
Epoch 10/20 - Train Accuracy: 0.9057
Epoch 15/20 - Train Accuracy: 0.9069
Epoch 20/20 - Train Accuracy: 0.9078


In [17]:
def macro_f1(y_true, y_pred, n_classes):

    f1_scores = []

    for c in range(n_classes):

        tp = np.sum(
            (y_true == c) & (y_pred == c)
        )

        fp = np.sum(
            (y_true != c) & (y_pred == c)
        )

        fn = np.sum(
            (y_true == c) & (y_pred != c)
        )

        precision = tp / (tp + fp + 1e-8)
        recall = tp / (tp + fn + 1e-8)

        f1 = (
            2 * precision * recall /
            (precision + recall + 1e-8)
        )

        f1_scores.append(f1)

    return np.mean(f1_scores)

In [18]:
val_pred = model.predict(X_val)

val_f1 = macro_f1(
    y_val,
    val_pred,
    len(LANGUAGES)
)

print("Validation Macro-F1:", round(val_f1, 4))

Validation Macro-F1: 0.8923


In [19]:
test_pred = model.predict(X_test)

test_f1 = macro_f1(
    y_test,
    test_pred,
    len(LANGUAGES)
)

print("Test Macro-F1:", round(test_f1, 4))

Test Macro-F1: 0.8897


In [20]:
accuracy = np.mean(test_pred == y_test)

print("Test Accuracy:", round(accuracy, 4))
print("Test Macro-F1:", round(test_f1, 4))

Test Accuracy: 0.9009
Test Macro-F1: 0.8897


In [21]:
def per_class_f1(y_true, y_pred, n_classes):

    scores = []

    for c in range(n_classes):

        tp = np.sum((y_true == c) & (y_pred == c))
        fp = np.sum((y_true != c) & (y_pred == c))
        fn = np.sum((y_true == c) & (y_pred != c))

        precision = tp / (tp + fp + 1e-8)
        recall = tp / (tp + fn + 1e-8)

        f1 = 2 * precision * recall / (
            precision + recall + 1e-8
        )

        scores.append(f1)

    return scores


scores = per_class_f1(
    y_test,
    test_pred,
    len(LANGUAGES)
)

results = pd.DataFrame({
    "Language": [
        LANGUAGES[x] for x in label_names
    ],
    "F1": scores
})

results.sort_values(
    "F1",
    ascending=False
)

,Language,F1
5,Gujarati,1.000000
15,Punjabi,1.000000
19,Telugu,1.000000
22,Santali,1.000000
17,Sindhi,1.000000
18,Tamil,1.000000
10,Malayalam,1.000000
20,Urdu,0.995025
7,Kannada,0.995025
14,Odia,0.995025


In [22]:
print("=" * 50)
print("LANGUAGE IDENTIFICATION RESULTS")
print("=" * 50)

print("Languages       :", len(LANGUAGES))
print("Sentences/lang  : 1000")
print("Training        :", len(train_texts))
print("Validation      :", len(val_texts))
print("Testing         :", len(test_texts))
print("TF-IDF features :", len(vectorizer.vocab))
print("Test Accuracy   :", round(accuracy, 4))
print("Test Macro-F1   :", round(test_f1, 4))

LANGUAGE IDENTIFICATION RESULTS
Languages       : 23
Sentences/lang  : 1000
Training        : 18400
Validation      : 2300
Testing         : 2300
TF-IDF features : 12000
Test Accuracy   : 0.9009
Test Macro-F1   : 0.8897


In [23]:
df = pd.DataFrame({
    "text": all_texts,
    "label": all_labels
})

df.to_csv(
    "indic_language_dataset.csv",
    index=False,
    encoding="utf-8"
)

print("Dataset saved!")

Dataset saved!
